In [ ]:
# ── 1. GPU check ─────────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("✅ GPU detected")
else:
    print("⚠️ مفيش GPU!Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── 2. فحص الفيديو المرفوع أو رفع يدوي ──────────────────────────────────
import os, glob

VIDEO_DIR = "/content/videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

# Search for existing videos
search_paths = ["/content", VIDEO_DIR]
video_exts = ["*.mp4", "*.webm", "*.mkv", "*.mov", "*.avi"]
found = []
for path in search_paths:
    for ext in video_exts:
        found.extend(glob.glob(os.path.join(path, ext)))

if found:
    video_path = found[0]
    print(f"✅ Found existing video: {os.path.basename(video_path)}")
    print(f"   Size: {os.path.getsize(video_path)/1024/1024:.1f} MB")
else:
    print("📤 ارفع الفيديو من جهازك:")
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = os.path.join(VIDEO_DIR, name)
        with open(dest, "wb") as f:
            f.write(data)
        print(f"✅ Uploaded: {name}")
    video_path = os.path.join(VIDEO_DIR, list(uploaded.keys())[0])

# Move to videos dir if needed
if not video_path.startswith(VIDEO_DIR):
    dest = os.path.join(VIDEO_DIR, os.path.basename(video_path))
    os.replace(video_path, dest)
    video_path = dest

print(f"\n🎬 Video ready: {video_path}")

In [ ]:
# ── 3. استخراج الصوت ومقطع الاستنساخ ─────────────────────────────────────
import subprocess

AUDIO_DIR = "/content/audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

# Extract full audio
full_audio = os.path.join(AUDIO_DIR, "full_audio.wav")
subprocess.run([
    "ffmpeg", "-y", "-i", video_path,
    "-vn", "-acodec", "pcm_s16le", "-ar", "24000", "-ac", "1",
    full_audio
], check=True)
print(f"✅ Full audio extracted: {full_audio}")

# Extract 10s reference clip from 30s mark
ref_audio = "/content/reference_voice.wav"
subprocess.run([
    "ffmpeg", "-y", "-ss", "30", "-t", "10", "-i", full_audio,
    "-ar", "24000", "-ac", "1",
    ref_audio
], check=True)
print(f"✅ Reference voice ready: {ref_audio}")
print("   📢 10-second clip for voice cloning")

In [ ]:
# ── 4. تثبيت VoiceStudio ─────────────────────────────────────────────────
import os, shutil, subprocess, sys

REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, cwd=None, what=""):
    print(f"\n$ {cmd if isinstance(cmd, str) else ' '.join(cmd)}")
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"❌ FAILED: {what} (exit {rc})")

# Clone/update repo
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Updating VoiceStudio repo...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    run(["git", "clone", "--depth", "1", "https://github.com/debpalash/VoiceStudio.git", REPO_DIR], what="git clone")

# System deps
run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get")

# Bun
if not os.path.exists(BUN):
    run("curl -fsSL https://bun.sh/install | bash", what="bun install")
os.environ["PATH"] = os.path.dirname(BUN) + os.pathsep + os.environ["PATH"]
run([BUN, "--version"], what="bun check")

# Frontend build
dist = os.path.join(REPO_DIR, "frontend", "dist", "index.html")
if not os.path.exists(dist):
    run([BUN, "install", "--frozen-lockfile"], cwd=REPO_DIR, what="bun install")
    run([BUN, "run", "--cwd", "electron", "build:web"], cwd=REPO_DIR, what="build frontend")
else:
    print("Frontend already built")

# Backend deps
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip uv")
run([
    "uv", "pip", "install", "--system", "--no-cache",
    "--constraint", "deploy/torch-constraints.txt",
    "--override", "deploy/uv-overrides.txt",
    "."
], cwd=REPO_DIR, what="backend install")

# cuDNN compat
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")], what="cuDNN setup")

# Sanity check
run([sys.executable, "-c",
    "import torch, torchaudio; print(f'torch {torch.__version__}, torchaudio {torchaudio.__version__}, CUDA: {torch.cuda.is_available()}'); "
    "from omnivoice.models.omnivoice import OmniVoice; "
    "print('✅ VoiceStudio ready')"
], cwd=REPO_DIR, what="sanity check")

print("\n✅ VoiceStudio installed!")

In [ ]:
# ── 5. إعداد البيانات المحلية ────────────────────────────────────────────
DATA_DIR = "/content/omnivoice_data"
os.environ['OMNIVOICE_DATA_DIR'] = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'outputs'), exist_ok=True)

# Copy reference audio
ref_dest = os.path.join(DATA_DIR, 'reference_voice.wav')
if os.path.exists(ref_audio):
    shutil.copy(ref_audio, ref_dest)
    print(f"✅ Reference voice: {ref_dest}")

print(f"✅ Data dir: {DATA_DIR}")
print("✅ Using local storage (no Google Drive)")

In [ ]:
# ── 6. تشغيل Backend ─────────────────────────────────────────────────────
import requests, time

BACKEND_PORT = 3900
BASE = f"http://127.0.0.1:{BACKEND_PORT}"
LOG_FILE = "/content/omnivoice_backend.log"

# Ensure data dir
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join(DATA_DIR, 'outputs'), exist_ok=True)

# Start backend
backend_proc = subprocess.Popen(
    [sys.executable, "-m", "omnivoice.backend", "--port", str(BACKEND_PORT), "--data-dir", DATA_DIR],
    cwd=REPO_DIR,
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT,
    text=True
)
print(f"Backend PID: {backend_proc.pid}")

# Wait
print("Waiting for backend...")
for i in range(90):
    try:
        r = requests.get(f"{BASE}/health", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Backend up! {BASE}")
            break
    except:
        pass
    time.sleep(2)
else:
    print("\n❌ Backend failed. Log:")
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as f:
            for line in f.readlines()[-30:]:
                print(line.rstrip())
    raise SystemExit("Backend failed — see log above")

print("Health:", requests.get(f"{BASE}/health").json())

In [ ]:
# ── 7. واجهة الدبلجة ─────────────────────────────────────────────────────
import gradio as gr

def auto_dub(lang_choice):
    import requests, json, time
    lang_code = "ar" if "فصحى" in lang_choice else "ar-EG"

    print("📤 Uploading video to backend...")
    with open(video_path, "rb") as f:
        r = requests.post(
            f"{BASE}/dub/upload",
            files={"video": (os.path.basename(video_path), f, "video/mp4")},
            data={"input_type": "video"},
            timeout=300
        )
    if r.status_code != 202:
        return None, f"❌ Upload failed: {r.text[:200]}"
    job_id = r.json()["job_id"]
    print(f"✅ Job: {job_id}")

    print("🎧 Transcribing...")
    r = requests.post(f"{BASE}/dub/transcribe/{job_id}", timeout=1800)
    if r.status_code != 200:
        return None, f"❌ Transcription failed: {r.text[:200]}"
    segments = r.json()["segments"]
    print(f"✅ {len(segments)} segments")

    print("🌍 Translating...")
    r = requests.post(f"{BASE}/dub/translate", json={
        "job_id": job_id,
        "target_lang": lang_code,
        "provider": "google",
        "segments": [{
            "id": str(s["id"]), "text": s["text"],
            "start": s["start"], "end": s["end"]
        } for s in segments]
    }, timeout=600)
    if r.status_code != 200:
        return None, f"❌ Translation failed: {r.text[:200]}"
    translated = {t["id"]: t["text"] for t in r.json()["translated"]}
    print(f"✅ Translated {len(translated)} segments")

    print("🎙️ Generating Arabic audio...")
    r = requests.post(f"{BASE}/dub/generate/{job_id}", json={
        "language": "Arabic",
        "language_code": lang_code,
        "segments": [{
            "start": s["start"], "end": s["end"],
            "text": translated.get(str(s["id"]), s["text"])
        } for s in segments],
        "voice_clone": True,
        "reference_audio": ref_dest
    }, timeout=120)
    if r.status_code != 200:
        return None, f"❌ Audio generation failed: {r.text[:200]}"

    task_id = r.json()["task_id"]
    print(f"🎨 Rendering audio (task {task_id})...")
    for _ in range(120):
        time.sleep(5)
        r = requests.get(f"{BASE}/jobs/{task_id}", timeout=30)
        if r.status_code == 200:
            st = r.json().get("status")
            if st == "done":
                break
            if st in ("failed", "cancelled"):
                return None, f"❌ Dubbing {st}: {r.text[:200]}"
    else:
        return None, "❌ Dubbing timed out"

    print("📥 Downloading dubbed video...")
    r = requests.get(f"{BASE}/dub/download/{job_id}",
        params={"include_tracks": lang_code, "default_track": lang_code},
        timeout=1800)
    if r.status_code != 200:
        return None, f"❌ Download failed: {r.text[:200]}"

    out_path = os.path.join(DATA_DIR, "outputs", "dubbed_video.mp4")
    with open(out_path, "wb") as f:
        f.write(r.content)
    print(f"\n🎉 Done! {out_path}")
    return out_path, "✅ تمت الدبلجة! الفيديو جاهز"

with gr.Blocks(title="🇪🇬 Arabic Dub", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇪🇬 دبلجة الفيديو للعربية\n\n🎬 الفيديو جاهز للدبلجة")
    lang_choice = gr.Radio(choices=["ar (فصحى)", "ar-EG (مصرية)"], value="ar (فصحى)", label="اللغة")
    dub_btn = gr.Button("🚀 بدء الدبلجة", variant="primary", size="lg")
    output_video = gr.Video(label="🎬 الفيديو المدبلج")
    status = gr.Textbox(label="الحالة", interactive=False, lines=3)

    def on_dub(lang):
        status.value = "⏳ جاري المعالجة... (10-15 دقيقة)"
        out, msg = auto_dub(lang)
        if out:
            return out, msg
        return None, msg

    dub_btn.click(on_dub, inputs=[lang_choice], outputs=[output_video, status])

print("\n" + "="*60)
print("🚀 Launching interface...")
demo.launch(share=True, debug=True)